# Phase 2B: Classification (Predicting Categories)

## 🎯 Learning Objectives

By the end of this notebook, you will:

- ✅ Master classification algorithms
- ✅ Build Logistic Regression, KNN, Naive Bayes models
- ✅ Implement Decision Trees, Random Forest, SVM
- ✅ Understand classification metrics
- ✅ Build complete classification pipelines

**Time Required:** 1-2 weeks  
**Difficulty:** Intermediate  
**Prerequisites:** Phase 2A Regression completed

## 📊 What is Classification?

**Goal:** Predict which category (class) something belongs to.

**Examples:**
- 📧 Email: Spam or Not Spam
- 🏥 Patient: Disease or Healthy
- 💳 Transaction: Fraud or Legitimate
- 🌦️ Weather: Sunny, Rainy, or Cloudy
- 🐱 Image: Cat, Dog, or Bird

In [1]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, roc_curve, auc)

# Classification algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Datasets
from sklearn.datasets import make_classification, load_iris, load_breast_cancer

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---

## 2.7 Logistic Regression

### Concept
Despite the name, it's for **classification**, not regression!

Predicts probability of belonging to a class using sigmoid function.

**Formula:** $P(y=1) = \frac{1}{1 + e^{-(mx + b)}}$

**Output:** Probability between 0 and 1

In [ ]:
# Understanding the Sigmoid Function

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 100)
y = sigmoid(z)

plt.figure(figsize=(10, 6))
plt.plot(z, y, 'b-', linewidth=3)
plt.axhline(y=0.5, color='r', linestyle='--', label='Decision boundary (0.5)')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('z (linear combination)', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('Sigmoid Function', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Annotate
plt.annotate('Probability > 0.5\n→ Class 1', xy=(3, 0.95), fontsize=10, color='green')
plt.annotate('Probability < 0.5\n→ Class 0', xy=(-7, 0.05), fontsize=10, color='red')

plt.tight_layout()
plt.show()

print("Sigmoid converts any number to a probability (0-1)")
print(f"sigmoid(-5) = {sigmoid(-5):.4f} → Class 0")
print(f"sigmoid(0)  = {sigmoid(0):.4f} → Uncertain")
print(f"sigmoid(5)  = {sigmoid(5):.4f} → Class 1")

In [ ]:
# Binary Classification: Disease Prediction

# Generate data: Age vs Disease (0=Healthy, 1=Diseased)
np.random.seed(42)

# Healthy people (younger, lower risk)
healthy_age = np.random.normal(35, 10, 100)
healthy_labels = np.zeros(100)

# Diseased people (older, higher risk)
diseased_age = np.random.normal(55, 10, 100)
diseased_labels = np.ones(100)

# Combine
X = np.concatenate([healthy_age, diseased_age]).reshape(-1, 1)
y = np.concatenate([healthy_labels, diseased_labels])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)  # Probabilities

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"=== Logistic Regression Results ===")
print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Diseased']))

In [ ]:
# Visualize Logistic Regression

plt.figure(figsize=(14, 5))

# Plot 1: Data with probability curve
plt.subplot(1, 2, 1)
plt.scatter(X_train[y_train==0], y_train[y_train==0], color='blue', 
            label='Healthy', alpha=0.6, s=50)
plt.scatter(X_train[y_train==1], y_train[y_train==1], color='red', 
            label='Diseased', alpha=0.6, s=50)

# Probability curve
X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_plot_proba = model.predict_proba(X_plot)[:, 1]
plt.plot(X_plot, y_plot_proba, color='green', linewidth=3, label='Probability curve')
plt.axhline(y=0.5, color='black', linestyle='--', label='Decision boundary')

plt.xlabel('Age', fontsize=12)
plt.ylabel('Probability of Disease', fontsize=12)
plt.title('Logistic Regression: Disease Prediction', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Plot 2: Confusion Matrix
plt.subplot(1, 2, 2)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Healthy', 'Diseased'],
            yticklabels=['Healthy', 'Diseased'])
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title(f'Confusion Matrix\nAccuracy: {accuracy*100:.1f}%', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Predict for new patients
new_patients = np.array([[30], [45], [60], [70]])
predictions = model.predict(new_patients)
probabilities = model.predict_proba(new_patients)

print("=== New Patient Predictions ===")
for age, pred, prob in zip(new_patients, predictions, probabilities):
    status = "Diseased" if pred == 1 else "Healthy"
    confidence = prob[int(pred)] * 100
    print(f"Age {age[0]:2d}: {status:10s} (Confidence: {confidence:.1f}%)")

---

## 2.8 K-Nearest Neighbors (KNN)

### Concept
Classify based on the K nearest training examples.

**How it works:**
1. Find K nearest neighbors to new point
2. Take majority vote
3. Assign that class

**Analogy:** "You are the average of your 5 closest friends"

In [ ]:
# KNN Classification Example

# Generate 2D data for visualization
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1,
                          random_state=42)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train with different K values
k_values = [1, 3, 5, 10, 20]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, k in enumerate(k_values):
    # Train
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    # Accuracy
    train_acc = knn.score(X_train, y_train)
    test_acc = knn.score(X_test, y_test)
    
    # Plot decision boundary
    ax = axes[idx]
    
    # Create mesh grid
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdYlBu', 
               edgecolors='black', s=50, alpha=0.6)
    ax.set_title(f'K={k}\nTrain: {train_acc:.2f} | Test: {test_acc:.2f}', 
                fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Remove empty subplot
axes[5].axis('off')

plt.suptitle('KNN: Effect of K on Decision Boundary', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Small K (K=1): Complex boundary, may overfit")
print("- Large K: Smoother boundary, may underfit")
print("- Choose K using cross-validation!")

In [ ]:
# Find Optimal K using Cross-Validation

k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X, y, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Plot
plt.figure(figsize=(12, 6))
plt.plot(k_range, cv_scores, 'bo-', linewidth=2, markersize=6)
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('KNN: Choosing Optimal K', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Mark optimal K
best_k = k_range[np.argmax(cv_scores)]
best_score = max(cv_scores)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.scatter([best_k], [best_score], c='red', s=200, zorder=5)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Optimal K: {best_k}")
print(f"Best CV Accuracy: {best_score:.4f}")

---

## 2.9 Naive Bayes

### Concept
Uses Bayes' theorem with "naive" assumption of feature independence.

**Formula:** $P(Class|Features) = \frac{P(Features|Class) \times P(Class)}{P(Features)}$

**Best for:** Text classification, spam detection

In [ ]:
# Naive Bayes: Iris Classification

# Load iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print("Iris Dataset:")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")
print(f"Shape: {X.shape}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)

# Predictions
y_pred = nb.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"\n=== Naive Bayes Results ===")
print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Visualize Naive Bayes Confusion Matrix

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title(f'Naive Bayes: Iris Classification\nAccuracy: {accuracy*100:.1f}%', 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 2.10 Decision Trees

### Concept
Tree-like model of decisions. At each node, asks a question about a feature.

**Advantages:**
- Easy to understand and interpret
- Handles both numerical and categorical data
- No feature scaling needed

**Disadvantages:**
- Can easily overfit
- Unstable (small data changes → big tree changes)

In [ ]:
# Decision Tree Classification

# Use breast cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print("Breast Cancer Dataset:")
print(f"Features: {len(cancer.feature_names)} features")
print(f"Classes: {cancer.target_names}")
print(f"Samples: {len(y)} ({sum(y)} benign, {len(y)-sum(y)} malignant)")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

# Evaluate
y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n=== Decision Tree Results ===")
print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Visualize Feature Importance

# Get feature importances
importances = dt.feature_importances_
indices = np.argsort(importances)[::-1][:10]  # Top 10

plt.figure(figsize=(12, 6))
plt.barh(range(10), importances[indices][::-1], align='center')
plt.yticks(range(10), [cancer.feature_names[i] for i in indices][::-1])
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Decision Tree: Top 10 Important Features', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:")
for i in range(5):
    print(f"  {i+1}. {cancer.feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

In [ ]:
# Effect of max_depth on Decision Tree

depths = range(1, 21)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

plt.figure(figsize=(12, 6))
plt.plot(depths, train_scores, 'bo-', label='Train', linewidth=2, markersize=6)
plt.plot(depths, test_scores, 'ro-', label='Test', linewidth=2, markersize=6)
plt.xlabel('Max Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Decision Tree: Effect of Max Depth', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Mark optimal depth
best_depth = depths[np.argmax(test_scores)]
plt.axvline(x=best_depth, color='green', linestyle='--', label=f'Best depth={best_depth}')

plt.tight_layout()
plt.show()

print(f"Best max_depth: {best_depth}")
print(f"Train accuracy at best depth: {train_scores[best_depth-1]:.4f}")
print(f"Test accuracy at best depth: {test_scores[best_depth-1]:.4f}")

---

## 2.11 Random Forest

### Concept
Ensemble of many decision trees. Each tree votes, majority wins!

**How it works:**
1. Create many decision trees with random subsets of data and features
2. Each tree makes a prediction
3. Final prediction = majority vote (classification) or average (regression)

**Advantages:**
- Reduces overfitting
- More accurate than single tree
- Handles missing values well

In [ ]:
# Random Forest Classification

# Use same breast cancer dataset
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=== Random Forest Results ===")
print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Compare Decision Tree vs Random Forest

# Train both
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

dt_acc = dt.score(X_test, y_test)
rf_acc = rf.score(X_test, y_test)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
ax1 = axes[0]
models = ['Decision Tree', 'Random Forest']
accuracies = [dt_acc, rf_acc]
colors = ['skyblue', 'lightgreen']
bars = ax1.bar(models, accuracies, color=colors, edgecolor='black', linewidth=2)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Accuracy Comparison', fontsize=13, fontweight='bold')
ax1.set_ylim(0.9, 1.0)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{acc:.3f}', ha='center', fontsize=12, fontweight='bold')

# Feature importance comparison
ax2 = axes[1]
top_n = 10
dt_imp = dt.feature_importances_
rf_imp = rf.feature_importances_
indices = np.argsort(rf_imp)[::-1][:top_n]

x = np.arange(top_n)
width = 0.35

ax2.barh(x - width/2, dt_imp[indices][::-1], width, label='Decision Tree', alpha=0.8)
ax2.barh(x + width/2, rf_imp[indices][::-1], width, label='Random Forest', alpha=0.8)
ax2.set_yticks(x)
ax2.set_yticklabels([cancer.feature_names[i][:20] for i in indices][::-1], fontsize=9)
ax2.set_xlabel('Feature Importance', fontsize=12)
ax2.set_title('Top 10 Feature Importance', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Decision Tree Accuracy: {dt_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"Improvement: {(rf_acc - dt_acc) * 100:.2f}%")

---

## 2.12 Support Vector Machines (SVM)

### Concept
Find the hyperplane that best separates classes with maximum margin.

**Key Ideas:**
- **Margin:** Distance between hyperplane and nearest data points
- **Support Vectors:** Data points closest to the hyperplane
- **Kernel Trick:** Transform data to higher dimensions for non-linear separation

In [ ]:
# SVM Classification with Different Kernels

# Generate non-linearly separable data
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Different kernels
kernels = ['linear', 'poly', 'rbf']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, kernel in zip(axes, kernels):
    # Train SVM
    svm = SVC(kernel=kernel, C=1.0, gamma='scale')
    svm.fit(X_train_scaled, y_train)
    
    # Accuracy
    acc = svm.score(X_test_scaled, y_test)
    
    # Plot decision boundary
    h = 0.02
    x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
    y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, 
               cmap='RdYlBu', edgecolors='black', s=50)
    ax.set_title(f'SVM Kernel: {kernel}\nTest Accuracy: {acc:.3f}', 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('SVM: Effect of Different Kernels', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Kernel Summary:")
print("- Linear: Best for linearly separable data")
print("- Polynomial: Adds polynomial features")
print("- RBF (Radial Basis Function): Most flexible, works for most cases")

---

## 2.13 Classification Metrics Deep Dive

In [ ]:
# Understanding Classification Metrics

# Use breast cancer dataset
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=== Confusion Matrix Explained ===")
print(f"\n{cm}\n")
print(f"True Negatives (TN): {tn} - Correctly predicted Malignant")
print(f"False Positives (FP): {fp} - Malignant predicted as Benign")
print(f"False Negatives (FN): {fn} - Benign predicted as Malignant")
print(f"True Positives (TP): {tp} - Correctly predicted Benign")

# Metrics
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)  # Of all positive predictions, how many are correct?
recall = tp / (tp + fn)     # Of all actual positives, how many did we catch?
f1 = 2 * (precision * recall) / (precision + recall)

print(f"\n=== Metrics ===")
print(f"Accuracy:  {accuracy:.4f}  (Overall correctness)")
print(f"Precision: {precision:.4f}  (When we say positive, are we right?)")
print(f"Recall:    {recall:.4f}  (Did we find all positives?)")
print(f"F1 Score:  {f1:.4f}  (Harmonic mean of precision & recall)")

In [ ]:
# ROC Curve and AUC

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, 'b-', linewidth=3, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random classifier')
plt.fill_between(fpr, tpr, alpha=0.2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Annotate
plt.annotate('Perfect classifier\n(top-left corner)', xy=(0, 1), xytext=(0.2, 0.8),
             arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green')

plt.tight_layout()
plt.show()

print("ROC Curve Interpretation:")
print(f"- AUC = {roc_auc:.3f} (closer to 1 = better)")
print("- Perfect classifier: AUC = 1.0")
print("- Random classifier: AUC = 0.5")
print("- Useless classifier: AUC < 0.5")

---

## 2.14 Compare All Classification Algorithms

In [ ]:
# Compare All Classification Algorithms

# Use breast cancer dataset
X = cancer.data
y = cancer.target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

# All classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

# Train and evaluate all
results = []

print("=== Classification Algorithm Comparison ===")
print(f"{'Algorithm':<25} {'Accuracy':>12} {'Precision':>12} {'Recall':>12} {'F1':>12}")
print("-" * 75)

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        'Algorithm': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })
    
    print(f"{name:<25} {acc:>12.4f} {prec:>12.4f} {rec:>12.4f} {f1:>12.4f}")

# Find best
results_df = pd.DataFrame(results)
best_idx = results_df['F1'].idxmax()
print(f"\n🏆 Best Algorithm: {results_df.loc[best_idx, 'Algorithm']} (F1: {results_df.loc[best_idx, 'F1']:.4f})")

In [ ]:
# Visualize Comparison

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(classifiers))
width = 0.2

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
colors = ['steelblue', 'darkorange', 'forestgreen', 'crimson']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Algorithm Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Algorithm'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim(0.9, 1.02)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

## 📝 Practice Exercises

### Exercise 1: Build a Spam Classifier

In [ ]:
# Exercise 1: Build a simple spam classifier

# Create synthetic spam data
np.random.seed(42)

# Features: [word_count, link_count, exclamation_count, caps_ratio]
# Not spam emails
not_spam = np.random.normal([50, 0.5, 1, 0.05], [20, 0.3, 1, 0.03], (100, 4))
not_spam_labels = np.zeros(100)

# Spam emails
spam = np.random.normal([30, 3, 5, 0.3], [15, 1, 2, 0.1], (100, 4))
spam_labels = np.ones(100)

# Combine
X = np.vstack([not_spam, spam])
y = np.concatenate([not_spam_labels, spam_labels])

# Ensure non-negative values
X = np.abs(X)

print("Spam Classification Dataset:")
print(f"Features: word_count, link_count, exclamation_count, caps_ratio")
print(f"Samples: {len(y)} (100 not spam, 100 spam)")

# TODO: 
# 1. Split data into train/test
# 2. Train at least 3 different classifiers
# 3. Compare their performance
# 4. Which one works best? Why?

print("\nComplete the exercise!")

---

## ✅ Phase Completion Checklist

- [ ] Understand Logistic Regression and sigmoid function
- [ ] Implement K-Nearest Neighbors with optimal K selection
- [ ] Apply Naive Bayes for classification
- [ ] Build Decision Trees and understand feature importance
- [ ] Implement Random Forest as ensemble method
- [ ] Use SVM with different kernels
- [ ] Calculate and interpret all classification metrics
- [ ] Understand ROC curve and AUC
- [ ] Compare multiple algorithms and choose the best

---

## 🎯 Key Takeaways

1. **Logistic Regression**: Simple, interpretable, good baseline
2. **KNN**: Simple, no training, but slow for large datasets
3. **Naive Bayes**: Fast, works well for text, assumes feature independence
4. **Decision Trees**: Easy to interpret, but prone to overfitting
5. **Random Forest**: Reduces overfitting, very powerful
6. **SVM**: Works well in high dimensions, kernel trick for non-linear data
7. **Always use cross-validation** to compare models fairly
8. **Choose metrics wisely**: Accuracy isn't always the best metric!

---

## 📚 Next Steps

👉 **[Phase-3-Unsupervised-Learning.ipynb](Phase-3-Unsupervised-Learning.ipynb)** - Discover hidden patterns!

Learn to find structure in unlabeled data:
- K-Means Clustering
- Hierarchical Clustering
- DBSCAN
- Dimensionality Reduction (PCA, t-SNE)

**Happy Learning! 🚀**